# Coach DNA EDA

I’m using this notebook to check the upgraded analytical layer behind the project and make sure the logic holds up before I lean on the outputs too hard.

The first version of the model gave me a solid start. Down, distance, score, and clock all helped. But once I started reading through the tables, it was clear the model was still too broad. It could describe the game, but it still wasn’t capturing enough of how real football decisions get made.

Field position was the missing piece.

That pushed me to rebuild the model around **situation + field zone**. This notebook is where I check that upgrade, see what cleaned up, and figure out what’s worth carrying into the profile and scoring work.

## What I’m doing here
- make sure the exported feature and scoring tables loaded correctly
- check team, situation, and field-zone coverage
- make sure the field-zone split is actually improving the model
- look at how run/dropback behavior shifts across situations and field position
- spot patterns worth carrying into team profiles and scoring
- pull early findings that could matter for CPU-controlled coaching logic

In [86]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

candidate_paths = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None

for path in candidate_paths:
    if (path / "python").exists() and (path / "data").exists():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root from notebook.")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("OUTPUT_TABLES_DIR:", OUTPUT_TABLES_DIR)

PROJECT_ROOT: /Users/Tip/Desktop/ea-coach-dna-calibration
PROCESSED_DATA_DIR: /Users/Tip/Desktop/ea-coach-dna-calibration/data/processed
OUTPUT_TABLES_DIR: /Users/Tip/Desktop/ea-coach-dna-calibration/outputs/tables


In [87]:
team_baseline_features = pd.read_csv(
    PROCESSED_DATA_DIR / "team_baseline_features_2025_vs_2023_2025.csv"
)

coach_dna_situation_scores = pd.read_csv(
    PROCESSED_DATA_DIR / "coach_dna_situation_scores_2025.csv"
)

coach_dna_team_summary = pd.read_csv(
    PROCESSED_DATA_DIR / "coach_dna_team_summary_2025.csv"
)

ranked_team_summary = pd.read_csv(
    OUTPUT_TABLES_DIR / "coach_dna_ranked_team_summary_2025.csv"
)

ranked_situation_scores = pd.read_csv(
    OUTPUT_TABLES_DIR / "coach_dna_ranked_situation_scores_2025.csv"
)

top_signal_situations = pd.read_csv(
    OUTPUT_TABLES_DIR / "coach_dna_top_signal_situations_by_team_2025.csv"
)

situation_strength_summary = pd.read_csv(
    OUTPUT_TABLES_DIR / "coach_dna_situation_strength_summary_2025.csv"
)

print("team_baseline_features:", team_baseline_features.shape)
print("coach_dna_situation_scores:", coach_dna_situation_scores.shape)
print("coach_dna_team_summary:", coach_dna_team_summary.shape)
print("ranked_team_summary:", ranked_team_summary.shape)
print("ranked_situation_scores:", ranked_situation_scores.shape)
print("top_signal_situations:", top_signal_situations.shape)
print("situation_strength_summary:", situation_strength_summary.shape)

team_baseline_features: (2458, 65)
coach_dna_situation_scores: (2458, 65)
coach_dna_team_summary: (32, 17)
ranked_team_summary: (32, 18)
ranked_situation_scores: (2458, 27)
top_signal_situations: (96, 23)
situation_strength_summary: (78, 13)


## Why this notebook changed

The first pass of the project was built around the usual football buckets:
- down
- distance
- score state
- clock

Once I started checking outputs though, I could see the model was still blending together football situations that weren’t really the same. A team backed up on its own side of the field shouldn’t be treated like that same team working in fringe space or near the goal line.

So I rebuilt the model around **situation + field zone**.

That changes the questions I’m asking in this notebook. I’m not just asking how teams behave on early downs or when they’re trailing. I’m asking how they behave on early downs when backed up, how they behave when trailing in fringe space, and how they change once the field gets compressed.

## 1. Basic Structure Checks

Before I start interpreting anything, I want to make sure the tables loaded the way I expect.

At this stage I’m checking the basics:
- row counts
- team coverage
- situation coverage
- field-zone coverage
- whether the feature table and score tables are lined up at the right grain

In [88]:
team_baseline_features.head()

,profile_season,baseline_type,baseline_start_season,baseline_end_season,team,situation_order,situation_name,field_zone_order,field_zone,situation_field_zone_context,team_play_count,baseline_play_count,season_count,team_count,team_season_count,team_sample_quality,baseline_quality,sample_vs_baseline_context,meets_min_sample_50,meets_min_sample_20,dropback_rate_team,dropback_rate_baseline,dropback_rate_delta,rush_rate_team,rush_rate_baseline,rush_rate_delta,pass_attempt_rate_team,pass_attempt_rate_baseline,pass_attempt_rate_delta,shotgun_rate_team,shotgun_rate_baseline,shotgun_rate_delta,no_huddle_rate_team,no_huddle_rate_baseline,no_huddle_rate_delta,avg_yards_gained_team,avg_yards_gained_baseline,avg_yards_gained_delta,avg_epa_team,avg_epa_baseline,avg_epa_delta,success_rate_team,success_rate_baseline,success_rate_delta,first_down_rate_team,first_down_rate_baseline,first_down_rate_delta,touchdown_rate_team,touchdown_rate_baseline,touchdown_rate_delta,turnover_rate_team,turnover_rate_baseline,turnover_rate_delta,sack_rate_team,sack_rate_baseline,sack_rate_delta,explosive_play_rate_team,explosive_play_rate_baseline,explosive_play_rate_delta,explosive_dropback_rate_team,explosive_dropback_rate_baseline,explosive_dropback_rate_delta,explosive_run_rate_team,explosive_run_rate_baseline,explosive_run_rate_delta
0,2025,league_multi_season,2023,2025,ARI,1,all_offense,1,backed_up,all_offense | backed_up,83,7747,3,32,96,good,strong,good_team_sample_vs_strong_baseline,1,1,0.6386,0.5653,0.0733,0.3614,0.4347,-0.0733,0.6386,0.5653,0.0733,0.6867,0.6902,-0.0035,0.0482,0.0541,-0.0059,5.5904,6.0160,-0.4256,-0.0714,-0.0254,-0.0460,0.3614,0.3983,-0.0369,0.1687,0.2316,-0.0629,0.0000,0.0053,-0.0053,0.0120,0.0258,-0.0138,0.0120,0.0320,-0.0200,0.0964,0.1407,-0.0443,0.0482,0.0901,-0.0419,0.0482,0.0506,-0.0024
1,2025,league_multi_season,2023,2025,ARI,1,all_offense,2,own_territory,all_offense | own_territory,470,45668,3,32,96,strong,strong,strong_team_sample_vs_strong_baseline,1,1,0.6766,0.6020,0.0746,0.3234,0.3980,-0.0746,0.6766,0.6020,0.0746,0.6787,0.7042,-0.0255,0.0872,0.0973,-0.0101,5.5149,6.0183,-0.5034,-0.0295,-0.0121,-0.0174,0.4298,0.4335,-0.0037,0.2915,0.2915,0.0000,0.0043,0.0059,-0.0016,0.0191,0.0286,-0.0095,0.0681,0.0451,0.0230,0.1532,0.1510,0.0022,0.1128,0.0993,0.0135,0.0404,0.0517,-0.0113
2,2025,league_multi_season,2023,2025,ARI,1,all_offense,3,fringe,all_offense | fringe,352,31340,3,32,96,strong,strong,strong_team_sample_vs_strong_baseline,1,1,0.6307,0.5689,0.0618,0.3693,0.4311,-0.0618,0.6307,0.5689,0.0618,0.7358,0.7056,0.0302,0.1818,0.1304,0.0514,5.5852,5.6104,-0.0252,0.0434,0.0079,0.0355,0.4688,0.4510,0.0178,0.3182,0.3014,0.0168,0.0114,0.0231,-0.0117,0.0170,0.0269,-0.0099,0.0455,0.0384,0.0071,0.1307,0.1440,-0.0133,0.1023,0.0889,0.0134,0.0284,0.0552,-0.0268
3,2025,league_multi_season,2023,2025,ARI,1,all_offense,4,red_zone,all_offense | red_zone,165,15231,3,32,96,strong,strong,strong_team_sample_vs_strong_baseline,1,1,0.6909,0.5029,0.1880,0.3091,0.4971,-0.1880,0.6909,0.5029,0.1880,0.8000,0.6589,0.1411,0.1273,0.0945,0.0328,3.1455,3.0000,0.1455,-0.0824,-0.0022,-0.0802,0.3879,0.4224,-0.0345,0.3030,0.3130,-0.0100,0.2061,0.1961,0.0100,0.0303,0.0258,0.0045,0.0606,0.0328,0.0278,0.0727,0.0512,0.0215,0.0606,0.0238,0.0368,0.0121,0.0274,-0.0153
4,2025,league_multi_season,2023,2025,ARI,2,early_down,1,backed_up,early_down | backed_up,61,6165,3,32,96,good,strong,good_team_sample_vs_strong_baseline,1,1,0.5738,0.5084,0.0654,0.4262,0.4916,-0.0654,0.5738,0.5084,0.0654,0.5902,0.6247,-0.0345,0.0164,0.0482,-0.0318,5.4098,5.9092,-0.4994,-0.0225,-0.0218,-0.0007,0.3607,0.4052,-0.0445,0.1148,0.2107,-0.0959,0.0000,0.0047,-0.0047,0.0164,0.0229,-0.0065,0.0000,0.0206,-0.0206,0.0820,0.1328,-0.0508,0.0328,0.0796,-0.0468,0.0492,0.0532,-0.0040


In [89]:
coach_dna_situation_scores.head()

,profile_season,team,situation_order,situation_name,field_zone_order,field_zone,situation_field_zone_context,team_play_count,team_sample_quality,sample_vs_baseline_context,sample_reliability_score,sample_multiplier,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,coach_dna_score_raw,coach_dna_score_adjusted,tendency_profile_label,tempo_profile_label,formation_profile_label,efficiency_profile_label,dropback_rate_team,dropback_rate_baseline,dropback_rate_delta,rush_rate_team,rush_rate_baseline,rush_rate_delta,pass_attempt_rate_team,pass_attempt_rate_baseline,pass_attempt_rate_delta,shotgun_rate_team,shotgun_rate_baseline,shotgun_rate_delta,no_huddle_rate_team,no_huddle_rate_baseline,no_huddle_rate_delta,avg_epa_team,avg_epa_baseline,avg_epa_delta,success_rate_team,success_rate_baseline,success_rate_delta,explosive_play_rate_team,explosive_play_rate_baseline,explosive_play_rate_delta,turnover_rate_team,turnover_rate_baseline,turnover_rate_delta,sack_rate_team,sack_rate_baseline,sack_rate_delta,dropback_rate_signal_score,shotgun_rate_signal_score,no_huddle_rate_signal_score,pass_attempt_rate_signal_score,avg_epa_score,success_rate_score,first_down_rate_score,touchdown_rate_score,explosive_play_rate_score,explosive_dropback_rate_score,explosive_run_rate_score,turnover_rate_score,sack_rate_score
0,2025,ARI,1,all_offense,1,backed_up,all_offense | backed_up,83,good,good_team_sample_vs_strong_baseline,80.0,0.9,50.000000,26.562500,27.083333,81.25000,45.328125,40.795313,more_dropback_heavy_than_baseline,close_to_baseline_tempo,close_to_baseline_shotgun_usage,less_efficient_than_baseline,0.6386,0.5653,0.0733,0.3614,0.4347,-0.0733,0.6386,0.5653,0.0733,0.6867,0.6902,-0.0035,0.0482,0.0541,-0.0059,-0.0714,-0.0254,-0.0460,0.3614,0.3983,-0.0369,0.0964,0.1407,-0.0443,0.0120,0.0258,-0.0138,0.0120,0.0320,-0.0200,84.375,9.3750,21.875,84.375,40.625,15.625,12.500,37.5000,9.375,18.750,53.125,78.1250,84.375
1,2025,ARI,1,all_offense,2,own_territory,all_offense | own_territory,470,strong,strong_team_sample_vs_strong_baseline,100.0,1.0,53.906250,41.796875,48.958333,40.62500,51.113281,51.113281,more_dropback_heavy_than_baseline,close_to_baseline_tempo,close_to_baseline_shotgun_usage,close_to_baseline_efficiency,0.6766,0.6020,0.0746,0.3234,0.3980,-0.0746,0.6766,0.6020,0.0746,0.6787,0.7042,-0.0255,0.0872,0.0973,-0.0101,-0.0295,-0.0121,-0.0174,0.4298,0.4335,-0.0037,0.1532,0.1510,0.0022,0.0191,0.0286,-0.0095,0.0681,0.0451,0.0230,93.750,18.7500,9.375,93.750,31.250,40.625,53.125,42.1875,56.250,71.875,18.750,78.1250,3.125
2,2025,ARI,1,all_offense,3,fringe,all_offense | fringe,352,strong,strong_team_sample_vs_strong_baseline,100.0,1.0,54.296875,39.062500,39.583333,61.71875,51.308594,51.308594,more_dropback_heavy_than_baseline,faster_than_baseline,close_to_baseline_shotgun_usage,more_efficient_than_baseline,0.6307,0.5689,0.0618,0.3693,0.4311,-0.0618,0.6307,0.5689,0.0618,0.7358,0.7056,0.0302,0.1818,0.1304,0.0514,0.0434,0.0079,0.0355,0.4688,0.4510,0.0178,0.1307,0.1440,-0.0133,0.0170,0.0269,-0.0099,0.0455,0.0384,0.0071,78.125,20.3125,40.625,78.125,43.750,50.000,56.250,6.2500,37.500,71.875,9.375,89.0625,34.375
3,2025,ARI,1,all_offense,4,red_zone,all_offense | red_zone,165,strong,strong_team_sample_vs_strong_baseline,100.0,1.0,78.125000,38.281250,66.666667,15.62500,61.289062,61.289062,more_dropback_heavy_than_baseline,faster_than_baseline,more_shotgun_than_baseline,less_efficient_than_baseline,0.6909,0.5029,0.1880,0.3091,0.4971,-0.1880,0.6909,0.5029,0.1880,0.8000,0.6589,0.1411,0.1273,0.0945,0.0328,-0.0824,-0.0022,-0.0802,0.3879,0.4224,-0.0345,0.0727,0.0512,0.0215,0.0303,0.0258,0.0045,0.0606,0.0328,0.0278,100.000,71.8750,40.625,100.000,25.000,21.875,40.625,65.6250,84.375,100.000,15.625,25.0000,6.250
4,2025,ARI,2,early_down,1,backed_up,early_down | backed_up,61,good,good_team_sample_vs_strong_baseline,80.0,0.9,48.437500,28.515625,25.000000,71.87500,43.863281,39.476953,more_dropback_heavy_than_baseline,slower_than_baseline,clo

In [90]:
coach_dna_team_summary.head()

,profile_season,team,scored_situations,overall_coach_dna_score,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_situation,top_signal_field_zone,top_signal_context,top_signal_score,lowest_signal_situation,lowest_signal_field_zone,lowest_signal_context,lowest_signal_score
0,2025,LA,45,56.2940,62.4652,72.6318,65.3343,62.2810,73.6662,neutral_early_down,fringe,neutral_early_down | fringe,80.6250,two_minute_game,own_territory,two_minute_game | own_territory,17.2246
1,2025,BUF,45,53.1461,63.8664,63.7499,62.2197,47.1380,72.1871,leading_early_down,own_territory,leading_early_down | own_territory,83.5156,tied_early_down,backed_up,tied_early_down | backed_up,16.5801
2,2025,WAS,45,51.9778,73.1083,52.7241,59.8365,47.8622,69.8413,neutral_early_down,own_territory,neutral_early_down | own_territory,83.3984,two_minute_game,red_zone,two_minute_game | red_zone,23.4113
3,2025,CIN,46,50.8784,64.2256,60.0367,47.1958,53.3315,73.8247,trailing_one_score,fringe,trailing_one_score | fringe,71.6406,two_minute_game,backed_up,two_minute_game | backed_up,17.9487
4,2025,BAL,45,49.1322,61.2633,54.7651,63.6064,35.2844,71.8062,neutral_early_down,fringe,neutral_early_down | fringe,77.4219,trailing_early_down,backed_up,trailing_early_down | backed_up,11.8242


In [91]:
pd.Series({
    "feature_rows": len(team_baseline_features),
    "feature_teams": team_baseline_features["team"].nunique(),
    "feature_situations": team_baseline_features["situation_name"].nunique(),
    "feature_field_zones": team_baseline_features["field_zone"].nunique(),
    "situation_score_rows": len(coach_dna_situation_scores),
    "team_summary_rows": len(coach_dna_team_summary),
})

feature_rows            2458
feature_teams             32
feature_situations        22
feature_field_zones        4
situation_score_rows    2458
team_summary_rows         32
dtype: int64

## 2. Situation and Field-Zone Coverage

This is the first real check on the upgraded model.

I wanted the project to move past broad situation labels and carry field position with them. So this section is really about one thing: did the data structure get sharper, or did I just make it more complicated?

If the upgrade worked, I should be able to see clean coverage by team, by situation, and by field zone.

In [92]:
team_baseline_features.groupby("team").size().sort_values().head(10)

team
NYJ    74
DET    75
SEA    76
PIT    76
NE     76
GB     76
JAX    76
KC     77
TB     77
SF     77
dtype: int64

In [93]:
team_baseline_features.groupby("situation_name").size().sort_values()

situation_name
goal_line                    32
goal_to_go                   32
red_zone                     32
neutral_early_down           96
fourth_down                 104
leading_two_plus_scores     122
two_minute_game             123
two_minute_half             127
trailing_two_plus_scores    127
short_yardage               127
trailing_one_score          128
trailing_early_down         128
trailing                    128
tied_early_down             128
all_offense                 128
third_down                  128
leading_one_score           128
leading_early_down          128
leading                     128
early_down                  128
tied                        128
one_score                   128
dtype: int64

In [94]:
team_baseline_features.groupby("field_zone").size().sort_values()

field_zone
backed_up        575
own_territory    606
fringe           607
red_zone         670
dtype: int64

In [95]:
team_baseline_features.groupby(["situation_name", "field_zone"]).size().sort_values()

situation_name           field_zone
fourth_down              backed_up      9
two_minute_game          backed_up     28
leading_two_plus_scores  backed_up     29
short_yardage            backed_up     31
two_minute_half          backed_up     31
                                       ..
leading_one_score        fringe        32
                         backed_up     32
leading_early_down       red_zone      32
short_yardage            red_zone      32
two_minute_half          red_zone      32
Length: 78, dtype: int64

In [96]:
team_baseline_features.groupby(["situation_name", "field_zone"])["team_play_count"].agg(
    ["count", "min", "median", "mean", "max"]
).sort_values("mean")

,,count,min,median,mean,max
situation_name,field_zone,,,,,
fourth_down,backed_up,9,1,1.0,1.333333,3
short_yardage,backed_up,31,1,3.0,3.290323,8
two_minute_game,backed_up,28,1,4.0,3.535714,8
two_minute_half,backed_up,31,2,7.0,7.225806,13
two_minute_game,red_zone,31,2,7.0,7.387097,20
...,...,...,...,...,...,...
early_down,fringe,32,189,251.5,248.781250,311
one_score,own_territory,32,227,304.5,305.000000,403
all_offense,fringe,32,260,337.5,331.406250,401


### How I’m reading this section

I’m looking for clean coverage at the **team + situation + field-zone** level.

That matters because I’m not trying to build a generic football behavior model anymore. I’m trying to get closer to the context where real coach decisions happen.

A team backed up in minus territory shouldn’t look like that same team in own territory. A team in fringe space shouldn’t look the same as that team inside compressed scoring space. If the model can’t separate those environments cleanly, it’s still too blunt for coach-logic tuning.

## 3. Sample Size Review by Situation and Field Zone

Once the structure looks right, the next question is whether the signal is strong enough to trust.

Some contexts happen all the time. Others don’t. That’s normal. I want to know where the model is sitting on strong footing and where I need to be more careful with the conclusions.

In [97]:
sample_summary = (
    team_baseline_features
    .groupby(["situation_name", "field_zone"], as_index=False)
    .agg(
        avg_team_play_count=("team_play_count", "mean"),
        median_team_play_count=("team_play_count", "median"),
        min_team_play_count=("team_play_count", "min"),
        max_team_play_count=("team_play_count", "max"),
    )
    .sort_values(["avg_team_play_count", "situation_name", "field_zone"])
)

sample_summary

,situation_name,field_zone,avg_team_play_count,median_team_play_count,min_team_play_count,max_team_play_count
8,fourth_down,backed_up,1.333333,1.0,1,3
38,short_yardage,backed_up,3.290323,3.0,1,8
70,two_minute_game,backed_up,3.535714,4.0,1,8
74,two_minute_half,backed_up,7.225806,7.0,2,13
73,two_minute_game,red_zone,7.387097,7.0,2,20
...,...,...,...,...,...,...
5,early_down,fringe,248.781250,251.5,189,311
35,one_score,own_territory,305.000000,304.5,227,403
1,all_offense,fringe,331.406250,337.5,260,401
6,early_down,own_territory,357.375000,357.5,315,399


In [98]:
team_baseline_features["team_sample_quality"].value_counts()

team_sample_quality
thin         702
strong       623
good         576
very_thin    557
Name: count, dtype: int64

In [99]:
team_baseline_features.groupby(
    ["situation_name", "field_zone", "team_sample_quality"]
).size().unstack(fill_value=0)

team_sample_quality            good  strong  thin  very_thin
situation_name  field_zone                                  
all_offense     backed_up        31       1     0          0
                fringe            0      32     0          0
                own_territory     0      32     0          0
                red_zone          0      32     0          0
early_down      backed_up        29       0     3          0
...                             ...     ...   ...        ...
two_minute_game red_zone          0       0     1         30
two_minute_half backed_up         0       0     0         31
                fringe            8       0    24          0
                own_territory    15       0    17          0
                red_zone          0       0    18         14

[78 rows x 4 columns]

### Why I care about this

Not every team-context combination is going to have the same level of stability.

Some situations come with plenty of volume:
- all offense in own territory
- early down in own territory
- one-score offense in fringe space

Others are naturally thinner:
- fourth down backed up
- late two-minute situations backed up
- goal-line behavior

For this project, the best tuning candidates are the places where the model has three things at once:
- real separation from baseline
- behavior that looks meaningful
- enough sample to trust what I’m looking at

## 4. League Baseline Tendencies by Situation and Field Zone

Before I can talk about team identity, I need a clean read on what normal NFL behavior looks like in each context.

That’s what this section is doing. It gives me the baseline layer before I start layering in team-specific behavior.

In [100]:
baseline_view = (
    team_baseline_features[
        [
            "situation_order",
            "situation_name",
            "field_zone_order",
            "field_zone",
            "situation_field_zone_context",
            "dropback_rate_baseline",
            "rush_rate_baseline",
            "shotgun_rate_baseline",
            "no_huddle_rate_baseline",
            "avg_epa_baseline",
            "success_rate_baseline",
            "explosive_play_rate_baseline",
        ]
    ]
    .drop_duplicates()
    .sort_values(["situation_order", "field_zone_order"])
    .reset_index(drop=True)
)

baseline_view

,situation_order,situation_name,field_zone_order,field_zone,situation_field_zone_context,dropback_rate_baseline,rush_rate_baseline,shotgun_rate_baseline,no_huddle_rate_baseline,avg_epa_baseline,success_rate_baseline,explosive_play_rate_baseline
0,1,all_offense,1,backed_up,all_offense | backed_up,0.5653,0.4347,0.6902,0.0541,-0.0254,0.3983,0.1407
1,1,all_offense,2,own_territory,all_offense | own_territory,0.6020,0.3980,0.7042,0.0973,-0.0121,0.4335,0.1510
2,1,all_offense,3,fringe,all_offense | fringe,0.5689,0.4311,0.7056,0.1304,0.0079,0.4510,0.1440
3,1,all_offense,4,red_zone,all_offense | red_zone,0.5029,0.4971,0.6589,0.0945,-0.0022,0.4224,0.0512
4,2,early_down,1,backed_up,early_down | backed_up,0.5084,0.4916,0.6247,0.0482,-0.0218,0.4052,0.1328
...,...,...,...,...,...,...,...,...,...,...,...,...
73,21,leading_early_down,4,red_zone,leading_early_down | red_zone,0.3776,0.6224,0.5435,0.0691,0.0257,0.4110,0.0545
74,22,trailing_early_down,1,backed_up,trailing_early_down | backed_up,0.5932,0.4068,0.7065,0.0695,-0.0183,0.4211,0.1378
75,22,trailing_early_down,2,own_territory,trailing_early_down | own_territory,0.6055,0.3945,0.7043,0.1311,-0.0018,0.4422,0.1471
76,22,trailing_early_down,3,fringe,trailing_early_down | fringe,0.6056,0.3944,0.7280,0.2006,-0.0037,0.4457,0.1392


### How I’m reading this table

This table is my default behavioral reference point.

It shows what the league tends to do in each **situation + field-zone** context before I start asking how teams differ from that baseline.

The things I care about most here are:
- run/dropback balance
- shotgun usage
- tempo
- efficiency
- how those patterns shift as field position changes

If the baseline is too broad, the team-level comparisons won’t be very useful. I wanted the baseline itself to understand football space better before I started scoring anyone against it.

In [101]:
baseline_view.sort_values("dropback_rate_baseline", ascending=False)[
    [
        "situation_field_zone_context",
        "dropback_rate_baseline",
        "rush_rate_baseline",
    ]
].head(20)

,situation_field_zone_context,dropback_rate_baseline,rush_rate_baseline
9,third_down | own_territory,0.7933,0.2067
24,two_minute_half | own_territory,0.7894,0.2106
8,third_down | backed_up,0.7874,0.2126
12,fourth_down | backed_up,0.7692,0.2308
28,two_minute_game | own_territory,0.7444,0.2556
25,two_minute_half | fringe,0.7380,0.2620
10,third_down | fringe,0.7139,0.2861
27,two_minute_game | backed_up,0.6988,0.3012
64,trailing_two_plus_scores | fringe,0.6938,0.3062
62,trailing_two_plus_scores | backed_up,0.6893,0.3107


In [102]:
baseline_view.sort_values("avg_epa_baseline", ascending=False)[
    [
        "situation_field_zone_context",
        "avg_epa_baseline",
        "success_rate_baseline",
        "explosive_play_rate_baseline",
    ]
].head(20)

,situation_field_zone_context,avg_epa_baseline,success_rate_baseline,explosive_play_rate_baseline
14,fourth_down | fringe,0.2667,0.5436,0.1632
18,short_yardage | fringe,0.1322,0.6381,0.1243
13,fourth_down | own_territory,0.0984,0.5277,0.1231
15,fourth_down | red_zone,0.0922,0.5669,0.0380
57,leading_two_plus_scores | red_zone,0.0913,0.4250,0.0580
26,two_minute_half | red_zone,0.0597,0.4013,0.0448
19,short_yardage | red_zone,0.0481,0.5446,0.0211
68,tied_early_down | fringe,0.0441,0.4615,0.1480
45,leading | red_zone,0.0430,0.4200,0.0530
22,goal_line | red_zone,0.0380,0.5192,0.0000


## 5. Team Tendency Deviations by Context

This is where the field-zone rebuild starts to earn its keep.

At this point, I’m not just asking who looks more run-heavy or more dropback-heavy than league baseline. I’m trying to pin down where those differences show up and whether they’re strong enough to matter.

In [103]:
most_run_heavy = (
    team_baseline_features
    .sort_values("rush_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "field_zone",
            "situation_field_zone_context",
            "team_play_count",
            "rush_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_run_heavy

,team,situation_name,field_zone,situation_field_zone_context,team_play_count,rush_rate_delta,avg_epa_delta,success_rate_delta
1629,NE,fourth_down,backed_up,fourth_down | backed_up,1,0.7692,2.7850,0.5128
1644,NE,two_minute_game,backed_up,two_minute_game | backed_up,3,0.6988,-0.1523,-0.4710
104,ATL,two_minute_game,backed_up,two_minute_game | backed_up,4,0.6988,0.0414,-0.2210
2023,PIT,two_minute_game,backed_up,two_minute_game | backed_up,1,0.6988,0.1048,0.5290
1260,LA,two_minute_game,red_zone,two_minute_game | red_zone,2,0.6000,-2.2458,0.1217
107,ATL,two_minute_game,red_zone,two_minute_game | red_zone,6,0.6000,0.0596,0.1217
184,BAL,two_minute_game,red_zone,two_minute_game | red_zone,3,0.6000,-0.6987,-0.3783
1090,JAX,fourth_down,own_territory,fourth_down | own_territory,3,0.5938,2.4944,0.4723
1038,IND,tied,backed_up,tied | backed_up,1,0.5494,1.0547,0.6157
1066,IND,tied_early_down,backed_up,tied_early_down | backed_up,1,0.4855,1.0427,0.6074


In [104]:
most_dropback_heavy = (
    team_baseline_features
    .sort_values("dropback_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "field_zone",
            "situation_field_zone_context",
            "team_play_count",
            "dropback_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_dropback_heavy

,team,situation_name,field_zone,situation_field_zone_context,team_play_count,dropback_rate_delta,avg_epa_delta,success_rate_delta
324,CAR,short_yardage,backed_up,short_yardage | backed_up,1,0.5799,1.0051,0.4167
479,CIN,short_yardage,backed_up,short_yardage | backed_up,1,0.5799,-1.1710,-0.5833
1863,NYJ,short_yardage,backed_up,short_yardage | backed_up,1,0.5799,-0.9194,-0.5833
939,HOU,short_yardage,backed_up,short_yardage | backed_up,1,0.5799,1.4479,0.4167
1785,NYG,short_yardage,backed_up,short_yardage | backed_up,1,0.5799,1.9150,0.4167
401,CHI,short_yardage,backed_up,short_yardage | backed_up,2,0.5799,-0.4579,-0.5833
1401,LV,short_yardage,backed_up,short_yardage | backed_up,5,0.5799,-1.2232,-0.3833
2165,SF,short_yardage,backed_up,short_yardage | backed_up,4,0.5799,0.7407,0.1667
2317,TEN,fourth_down,own_territory,fourth_down | own_territory,9,0.4062,-0.9461,-0.3055
2026,PIT,two_minute_game,red_zone,two_minute_game | red_zone,4,0.4000,-1.0946,-0.1283


### How I’m reading this table

This table shows where teams drift furthest from league expectation in run-pass tendency.

That’s useful because this is where broad football behavior starts turning into team identity.

I’m mostly watching for three things:
- how big the tendency gap is
- whether the sample is strong enough to trust
- whether the efficiency side is supporting the behavior

The strongest tuning signals aren’t just the biggest deviations. They’re the deviations that hold up when I look at the rest of the context too.

In [105]:
most_no_huddle_heavy = (
    team_baseline_features
    .sort_values("no_huddle_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "field_zone",
            "situation_field_zone_context",
            "team_play_count",
            "no_huddle_rate_delta",
            "avg_epa_delta",
            "success_rate_delta",
        ]
    ]
)

most_no_huddle_heavy

,team,situation_name,field_zone,situation_field_zone_context,team_play_count,no_huddle_rate_delta,avg_epa_delta,success_rate_delta
2407,WAS,two_minute_game,backed_up,two_minute_game | backed_up,1,0.8842,0.5124,0.5290
2320,TEN,short_yardage,backed_up,short_yardage | backed_up,1,0.8403,0.2683,0.4167
2394,WAS,fourth_down,fringe,fourth_down | fringe,12,0.8294,0.3539,0.0397
2393,WAS,fourth_down,own_territory,fourth_down | own_territory,5,0.6723,0.2239,0.0723
2432,WAS,leading_one_score,fringe,leading_one_score | fringe,41,0.6642,-0.2559,-0.0563
2397,WAS,short_yardage,own_territory,short_yardage | own_territory,25,0.6544,0.0771,0.0742
2391,WAS,third_down,fringe,third_down | fringe,70,0.6443,-0.3179,0.0120
2433,WAS,leading_one_score,red_zone,leading_one_score | red_zone,10,0.6329,0.5248,0.1833
2440,WAS,trailing_one_score,fringe,trailing_one_score | fringe,101,0.6289,0.0446,-0.0076
2413,WAS,one_score,fringe,one_score | fringe,183,0.6177,-0.0262,0.0093


### Why this section got better after the rebuild

Before the field-zone upgrade, I could say a team looked more run-heavy or more pass-heavy in a broad situation.

Now I can get a lot more specific.

I can see teams leaning run-heavy on early downs in own territory. I can see teams getting more pass-oriented in fringe space. I can see tempo shifts that only show up in certain parts of the field.

That’s a better football read, and it’s a better starting point for CPU-controlled coach logic.

## 6. Efficiency Deviations by Context

Style by itself isn’t enough for this project.

I also want to know where teams are beating baseline, because behavior gets a lot more interesting when it’s both distinct and productive.

In [106]:
most_efficient = (
    team_baseline_features
    .sort_values("avg_epa_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "field_zone",
            "situation_field_zone_context",
            "team_play_count",
            "avg_epa_delta",
            "success_rate_delta",
            "explosive_play_rate_delta",
        ]
    ]
)

most_efficient

,team,situation_name,field_zone,situation_field_zone_context,team_play_count,avg_epa_delta,success_rate_delta,explosive_play_rate_delta
244,BUF,fourth_down,own_territory,fourth_down | own_territory,4,3.3928,0.4723,0.6269
1933,PHI,fourth_down,backed_up,fourth_down | backed_up,1,3.3186,0.5128,-0.1282
475,CIN,fourth_down,backed_up,fourth_down | backed_up,1,2.8976,0.5128,-0.1282
2316,TEN,fourth_down,backed_up,fourth_down | backed_up,1,2.7959,0.5128,0.8718
1629,NE,fourth_down,backed_up,fourth_down | backed_up,1,2.7850,0.5128,0.8718
1090,JAX,fourth_down,own_territory,fourth_down | own_territory,3,2.4944,0.4723,0.2102
2253,TB,two_minute_game,backed_up,two_minute_game | backed_up,2,2.2742,0.5290,0.8224
2249,TB,two_minute_half,backed_up,two_minute_half | backed_up,2,2.2222,0.5206,0.8363
92,ATL,fourth_down,red_zone,fourth_down | red_zone,5,2.1624,0.2331,-0.0380
1400,LV,fourth_down,red_zone,fourth_down | red_zone,5,2.0180,0.2331,-0.0380


In [107]:
most_successful = (
    team_baseline_features
    .sort_values("success_rate_delta", ascending=False)
    .head(20)[
        [
            "team",
            "situation_name",
            "field_zone",
            "situation_field_zone_context",
            "team_play_count",
            "success_rate_delta",
            "avg_epa_delta",
            "explosive_play_rate_delta",
        ]
    ]
)

most_successful

,team,situation_name,field_zone,situation_field_zone_context,team_play_count,success_rate_delta,avg_epa_delta,explosive_play_rate_delta
2334,TEN,two_minute_game,red_zone,two_minute_game | red_zone,6,0.6217,1.0045,0.1234
1569,MIN,two_minute_game,red_zone,two_minute_game | red_zone,3,0.6217,1.9953,0.2900
1746,NO,leading_two_plus_scores,backed_up,leading_two_plus_scores | backed_up,2,0.6216,0.5167,0.3550
2034,PIT,tied,backed_up,tied | backed_up,2,0.6157,0.2581,-0.1307
1038,IND,tied,backed_up,tied | backed_up,1,0.6157,1.0547,0.8693
501,CIN,tied,backed_up,tied | backed_up,3,0.6157,1.1708,0.5360
529,CIN,tied_early_down,backed_up,tied_early_down | backed_up,3,0.6074,1.1588,0.5399
2062,PIT,tied_early_down,backed_up,tied_early_down | backed_up,2,0.6074,0.2461,-0.1268
1066,IND,tied_early_down,backed_up,tied_early_down | backed_up,1,0.6074,1.0427,0.8732
1680,NE,trailing_two_plus_scores,red_zone,trailing_two_plus_scores | red_zone,1,0.5810,1.2971,-0.0526


### Why I’m checking this

Field position changes more than style. It changes what effective offense looks like.

A team can look average at a broad level and still show real strength in a smaller context like:
- goal-to-go
- trailing in own territory
- neutral early downs in fringe space

Those are the spots where a generic CPU coach can feel flat. I want to separate behavior that’s merely different from behavior that’s different and actually working.

## 7. Team Deep Dive by Situation and Field Zone

This is where I stop looking at the whole league and zoom in on one team at a time.

For this pass, I’m using Buffalo because the upgraded model already flags them near the top of the offensive Coach DNA rankings, and I want to see where that identity is actually showing up.

In [108]:
TEAM_CODE = "BUF"

team_view = (
    team_baseline_features
    .loc[team_baseline_features["team"] == TEAM_CODE]
    .sort_values(["situation_order", "field_zone_order"])
)

team_view[
    [
        "team",
        "situation_name",
        "field_zone",
        "situation_field_zone_context",
        "team_play_count",
        "dropback_rate_delta",
        "rush_rate_delta",
        "shotgun_rate_delta",
        "no_huddle_rate_delta",
        "avg_epa_delta",
        "success_rate_delta",
        "explosive_play_rate_delta",
    ]
]

,team,situation_name,field_zone,situation_field_zone_context,team_play_count,dropback_rate_delta,rush_rate_delta,shotgun_rate_delta,no_huddle_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta
232,BUF,all_offense,backed_up,all_offense | backed_up,79,-0.0463,0.0463,-0.1839,-0.0541,-0.1588,-0.0186,-0.0015
233,BUF,all_offense,own_territory,all_offense | own_territory,441,-0.0669,0.0669,-0.1827,-0.0474,0.2452,0.0812,0.0508
234,BUF,all_offense,fringe,all_offense | fringe,346,-0.0544,0.0544,-0.1796,-0.0870,0.0949,0.0230,0.0294
235,BUF,all_offense,red_zone,all_offense | red_zone,190,-0.0871,0.0871,-0.2484,0.0529,0.1341,0.0723,-0.0091
236,BUF,early_down,backed_up,early_down | backed_up,66,-0.0387,0.0387,-0.1853,-0.0482,-0.0699,-0.0113,0.0187
...,...,...,...,...,...,...,...,...,...,...,...,...
304,BUF,leading_early_down,red_zone,leading_early_down | red_zone,50,-0.1376,0.1376,-0.3235,0.0709,-0.0145,-0.0110,-0.0545
305,BUF,trailing_early_down,backed_up,trailing_early_down | backed_up,24,0.0735,-0.0735,-0.2065,-0.0695,0.1225,-0.0461,0.0705
306,BUF,trailing_early_down,own_territory,trailing_early_down | own_territory,168,-0.0281,0.0281,-0.1686,-0.0359,0.1155,0.0816,0.0017
307,BUF,trailing_early_down,fringe,trailing_early_down | fringe,126,-0.0897,0.0897,-0.1486,-0.1450,0.2160,0.0226,0.0195


### Why I built this section

This is the part of the notebook that gets closest to an actual tuning conversation.

I’m not asking how Buffalo compares with league average in some broad, vague sense. I’m asking how Buffalo looks against league average in specific parts of the field and in specific game contexts.

That’s the level where this project starts getting useful for:
- playcall weighting
- run/pass balance
- formation preference
- tempo
- scoring-space behavior
- protecting-a-lead versus chasing-points behavior

## 8. Score Sanity Checks

Before I pull bigger conclusions out of this, I want to make sure the scoring layer still lines up with the underlying deltas.

I’m checking whether the model is rewarding the kinds of patterns I actually care about:
- real separation from baseline
- context-specific behavior
- productive tendencies
- enough sample to trust what I’m seeing

In [109]:
coach_dna_team_summary.sort_values("overall_coach_dna_score", ascending=False).head(15)

,profile_season,team,scored_situations,overall_coach_dna_score,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_situation,top_signal_field_zone,top_signal_context,top_signal_score,lowest_signal_situation,lowest_signal_field_zone,lowest_signal_context,lowest_signal_score
0,2025,LA,45,56.2940,62.4652,72.6318,65.3343,62.2810,73.6662,neutral_early_down,fringe,neutral_early_down | fringe,80.6250,two_minute_game,own_territory,two_minute_game | own_territory,17.2246
1,2025,BUF,45,53.1461,63.8664,63.7499,62.2197,47.1380,72.1871,leading_early_down,own_territory,leading_early_down | own_territory,83.5156,tied_early_down,backed_up,tied_early_down | backed_up,16.5801
2,2025,WAS,45,51.9778,73.1083,52.7241,59.8365,47.8622,69.8413,neutral_early_down,own_territory,neutral_early_down | own_territory,83.3984,two_minute_game,red_zone,two_minute_game | red_zone,23.4113
3,2025,CIN,46,50.8784,64.2256,60.0367,47.1958,53.3315,73.8247,trailing_one_score,fringe,trailing_one_score | fringe,71.6406,two_minute_game,backed_up,two_minute_game | backed_up,17.9487
4,2025,BAL,45,49.1322,61.2633,54.7651,63.6064,35.2844,71.8062,neutral_early_down,fringe,neutral_early_down | fringe,77.4219,trailing_early_down,backed_up,trailing_early_down | backed_up,11.8242
5,2025,NE,45,49.0881,49.5704,68.2048,63.1267,44.2766,75.1637,leading_one_score,fringe,leading_one_score | fringe,66.8164,fourth_down,own_territory,fourth_down | own_territory,14.7419
6,2025,CHI,45,48.8438,50.1781,59.5375,61.3148,65.6443,74.7450,trailing_one_score,own_territory,trailing_one_score | own_territory,68.3203,two_minute_half,backed_up,two_minute_half | backed_up,12.8468
7,2025,SF,45,48.3072,55.0335,60.0075,47.3050,57.9523,74.2230,neutral_early_down,own_territory,neutral_early_down | own_territory,75.7227,fourth_down,own_territory,fourth_down | own_territory,11.9093
8,2025,SEA,44,47.6358,58.7242,56.5951,53.5932,47.8587,70.4528,third_down,own_territory,third_down | own_territory,70.9242,two_minute_half,red_zone,two_minute_half | red_zone,14.1875
9,2025,DET,43,46.0869,49.1242,58.7184,56.7840,55.7656,73.5606,leading_early_down,own_territory,leading_early_down | own_territory,68.3594,fourth_down,own_territory,fourth_down | own_territory,18.4718


In [110]:
coach_dna_situation_scores.sort_values("coach_dna_score_adjusted", ascending=False).head(20)[
    [
        "team",
        "situation_name",
        "field_zone",
        "situation_field_zone_context",
        "team_play_count",
        "coach_dna_score_adjusted",
        "tendency_signal_score",
        "efficiency_signal_score",
        "explosiveness_signal_score",
        "stability_signal_score",
        "tendency_profile_label",
        "efficiency_profile_label",
    ]
]

,team,situation_name,field_zone,situation_field_zone_context,team_play_count,coach_dna_score_adjusted,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,tendency_profile_label,efficiency_profile_label
274,BUF,leading,own_territory,leading | own_territory,154,87.890625,82.81250,96.875000,90.625000,78.12500,more_run_heavy_than_baseline,more_efficient_than_baseline
2386,WAS,early_down,own_territory,early_down | own_territory,344,84.960938,94.53125,85.937500,70.833333,53.12500,more_run_heavy_than_baseline,more_efficient_than_baseline
302,BUF,leading_early_down,own_territory,leading_early_down | own_territory,127,83.515625,78.90625,94.531250,80.208333,73.43750,more_run_heavy_than_baseline,more_efficient_than_baseline
2416,WAS,neutral_early_down,own_territory,neutral_early_down | own_territory,181,83.398438,95.31250,75.781250,87.500000,34.37500,more_run_heavy_than_baseline,more_efficient_than_baseline
187,BAL,one_score,fringe,one_score | fringe,211,83.125000,76.56250,98.437500,90.625000,54.68750,more_run_heavy_than_baseline,more_efficient_than_baseline
233,BUF,all_offense,own_territory,all_offense | own_territory,441,83.007812,78.90625,90.625000,88.541667,65.62500,more_run_heavy_than_baseline,more_efficient_than_baseline
157,BAL,all_offense,fringe,all_offense | fringe,282,82.617188,79.68750,97.656250,90.625000,37.50000,more_run_heavy_than_baseline,more_efficient_than_baseline
2382,WAS,all_offense,own_territory,all_offense | own_territory,421,82.460938,94.53125,70.312500,72.916667,64.06250,more_run_heavy_than_baseline,more_efficient_than_baseline
237,BUF,early_down,own_territory,early_down | own_territory,356,82.285156,81.25000,84.765625,83.333333,70.31250,more_run_heavy_than_baseline,more_efficient_than_baseline
466,CIN,all_offense,red_zone,all_offense | red_zone,146,82.070312,82.03125,90.625000,73.437500,64.84375,more_dropback_heavy_than_baseline,more_efficient_than_baseline


In [111]:
coach_dna_situation_scores.sort_values("coach_dna_score_adjusted", ascending=True).head(20)[
    [
        "team",
        "situation_name",
        "field_zone",
        "situation_field_zone_context",
        "team_play_count",
        "coach_dna_score_adjusted",
        "tendency_signal_score",
        "efficiency_signal_score",
        "explosiveness_signal_score",
        "stability_signal_score",
        "tendency_profile_label",
        "efficiency_profile_label",
    ]
]

,team,situation_name,field_zone,situation_field_zone_context,team_play_count,coach_dna_score_adjusted,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,tendency_profile_label,efficiency_profile_label
2240,TB,fourth_down,fringe,fourth_down | fringe,14,11.550781,7.812500,30.468750,20.833333,73.437500,close_to_baseline_run_pass_split,less_efficient_than_baseline
598,CLE,trailing_one_score,backed_up,trailing_one_score | backed_up,16,11.599609,7.812500,38.671875,25.000000,47.656250,close_to_baseline_run_pass_split,less_efficient_than_baseline
228,BAL,trailing_early_down,backed_up,trailing_early_down | backed_up,17,11.824219,12.890625,30.078125,46.354167,18.750000,close_to_baseline_run_pass_split,less_efficient_than_baseline
2162,SF,fourth_down,own_territory,fourth_down | own_territory,6,11.909274,14.919355,28.225806,30.107527,40.322581,more_dropback_heavy_than_baseline,less_efficient_than_baseline
408,CHI,two_minute_half,backed_up,two_minute_half | backed_up,7,12.846774,14.516129,30.645161,43.548387,34.677419,more_dropback_heavy_than_baseline,less_efficient_than_baseline
1491,MIA,two_minute_game,fringe,two_minute_game | fringe,20,13.034180,11.718750,18.359375,18.750000,19.531250,close_to_baseline_run_pass_split,less_efficient_than_baseline
1982,PHI,trailing_two_plus_scores,backed_up,trailing_two_plus_scores | backed_up,15,13.068548,25.000000,20.967742,33.870968,30.645161,close_to_baseline_run_pass_split,less_efficient_than_baseline
211,BAL,leading_two_plus_scores,red_zone,leading_two_plus_scores | red_zone,15,13.461694,23.387097,15.725806,27.419355,68.548387,close_to_baseline_run_pass_split,less_efficient_than_baseline
1029,IND,two_minute_game,fringe,two_minute_game | fringe,19,13.562500,39.453125,13.671875,14.062500,23.437500,close_to_baseline_run_pass_split,less_efficient_than_baseline
1552,MIN,fourth_down,own_territory,fourth_down | own_territory,8,13.723790,12.903226,44.758065,30.107527,44.354839,more_run_heavy_than_baseline,less_efficient_than_baseline


## 9. Draft Findings

This is where I start turning notebook work into project language.

I’m not trying to force polished conclusions too early. I’m trying to capture the strongest patterns the upgraded model is surfacing so I can carry the right ones into the README, GitHub write-up, and interview story.

In [113]:
display(situation_strength_summary.head(15))
display(ranked_team_summary.head(15))
display(top_signal_situations.head(20))

,rank,situation_order,situation_name,field_zone_order,field_zone,situation_field_zone_context,avg_adjusted_score,max_adjusted_score,min_adjusted_score,avg_team_play_count,team_count,strong_sample_teams,good_or_better_teams
0,1,1,all_offense,2,own_territory,all_offense | own_territory,53.9844,83.0078,25.7812,455.1,32,32,32
1,2,1,all_offense,3,fringe,all_offense | fringe,53.9844,82.6172,30.0781,331.4,32,32,32
2,3,1,all_offense,4,red_zone,all_offense | red_zone,53.9844,82.0703,17.5000,160.7,32,32,32
3,4,2,early_down,2,own_territory,early_down | own_territory,53.9844,84.9609,24.9805,357.4,32,32,32
4,5,2,early_down,3,fringe,early_down | fringe,53.9844,79.5703,23.7891,248.8,32,32,32
5,6,11,one_score,2,own_territory,one_score | own_territory,53.9844,81.7969,32.1094,305.0,32,32,32
6,7,11,one_score,3,fringe,one_score | fringe,53.9844,83.1250,30.1172,217.5,32,32,32
7,8,12,neutral_early_down,2,own_territory,neutral_early_down | own_territory,53.9844,83.3984,31.8945,230.7,32,32,32
8,9,15,trailing,2,own_territory,trailing | own_territory,53.9844,78.5547,29.6875,225.4,32,32,32
9,10,12,neutral_early_down,3,fringe,neutral_early_down | fringe,53.7940,80.6250,29.1211,155.1,32,31,32


,rank,team,overall_coach_dna_score,score_tier,scored_situations,tendency_signal_score_avg,efficiency_signal_score_avg,explosiveness_signal_score_avg,stability_signal_score_avg,sample_reliability_score_avg,top_signal_context,top_signal_situation,top_signal_field_zone,top_signal_score,lowest_signal_context,lowest_signal_situation,lowest_signal_field_zone,lowest_signal_score
0,1,LA,56.2940,strong_signal,45,62.4652,72.6318,65.3343,62.2810,73.6662,neutral_early_down | fringe,neutral_early_down,fringe,80.6250,two_minute_game | own_territory,two_minute_game,own_territory,17.2246
1,2,BUF,53.1461,solid_signal,45,63.8664,63.7499,62.2197,47.1380,72.1871,leading_early_down | own_territory,leading_early_down,own_territory,83.5156,tied_early_down | backed_up,tied_early_down,backed_up,16.5801
2,3,WAS,51.9778,solid_signal,45,73.1083,52.7241,59.8365,47.8622,69.8413,neutral_early_down | own_territory,neutral_early_down,own_territory,83.3984,two_minute_game | red_zone,two_minute_game,red_zone,23.4113
3,4,CIN,50.8784,solid_signal,46,64.2256,60.0367,47.1958,53.3315,73.8247,trailing_one_score | fringe,trailing_one_score,fringe,71.6406,two_minute_game | backed_up,two_minute_game,backed_up,17.9487
4,5,BAL,49.1322,solid_signal,45,61.2633,54.7651,63.6064,35.2844,71.8062,neutral_early_down | fringe,neutral_early_down,fringe,77.4219,trailing_early_down | backed_up,trailing_early_down,backed_up,11.8242
5,6,NE,49.0881,solid_signal,45,49.5704,68.2048,63.1267,44.2766,75.1637,leading_one_score | fringe,leading_one_score,fringe,66.8164,fourth_down | own_territory,fourth_down,own_territory,14.7419
6,7,CHI,48.8438,solid_signal,45,50.1781,59.5375,61.3148,65.6443,74.7450,trailing_one_score | own_territory,trailing_one_score,own_territory,68.3203,two_minute_half | backed_up,two_minute_half,backed_up,12.8468
7,8,SF,48.3072,solid_signal,45,55.0335,60.0075,47.3050,57.9523,74.2230,neutral_early_down | own_territory,neutral_early_down,own_territory,75.7227,fourth_down | own_territory,fourth_down,own_territory,11.9093
8,9,SEA,47.6358,solid_signal,44,58.7242,56.5951,53.5932,47.8587,70.4528,third_down | own_territory,third_down,own_territory,70.9242,two_minute_half | red_zone,two_minute_half,red_zone,14.1875
9,10,DET,46.0869,moderate_signal,43,49.1242,58.7184,56.7840,55.7656,73.5606,leading_early_down | own_territory,leading_early_down,own_territory,68.3594,fourth_down | own_territory,fourth_down,own_territory,18.4718


,team,team_rank_within_top_signals,situation_name,field_zone,situation_field_zone_context,team_play_count,team_sample_quality,coach_dna_score_adjusted,tendency_signal_score,efficiency_signal_score,explosiveness_signal_score,stability_signal_score,dropback_rate_delta,rush_rate_delta,shotgun_rate_delta,no_huddle_rate_delta,avg_epa_delta,success_rate_delta,explosive_play_rate_delta,tendency_profile_label,formation_profile_label,tempo_profile_label,efficiency_profile_label
0,ARI,1,early_down,red_zone,early_down | red_zone,118,strong,68.710938,78.906250,60.937500,61.458333,37.50000,0.2278,-0.2278,0.1794,0.0465,0.0980,-0.0269,0.0148,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
1,ARI,2,trailing_early_down,red_zone,trailing_early_down | red_zone,73,good,65.967188,85.937500,62.500000,66.145833,50.78125,0.2248,-0.2248,0.1377,0.1018,0.1736,-0.0038,0.0293,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
2,ARI,3,red_zone,red_zone,red_zone | red_zone,91,good,65.615625,89.062500,67.187500,68.750000,17.18750,0.2899,-0.2899,0.2410,0.0653,0.0322,-0.0325,0.0370,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
3,ATL,1,trailing_early_down,own_territory,trailing_early_down | own_territory,137,strong,74.804688,61.718750,78.125000,96.875000,79.68750,0.0514,-0.0514,0.1351,0.0514,0.2329,0.1052,0.1011,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
4,ATL,2,trailing,own_territory,trailing | own_territory,173,strong,73.613281,67.968750,65.234375,91.666667,79.68750,0.0589,-0.0589,0.1190,0.0673,0.0905,0.0731,0.0763,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,faster_than_baseline,more_efficient_than_baseline
5,ATL,3,trailing_one_score,own_territory,trailing_one_score | own_territory,88,good,66.600000,69.531250,77.343750,91.666667,56.25000,0.1072,-0.1072,0.1380,0.0052,0.3592,0.1524,0.1323,more_dropback_heavy_than_baseline,more_shotgun_than_baseline,close_to_baseline_tempo,more_efficient_than_baseline
6,BAL,1,one_score,fringe,one_score | fringe,211,strong,83.125000,76.562500,98.437500,90.625000,54.68750,-0.0954,0.0954,-0.0668,-0.0807,0.2823,0.0756,0.0537,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
7,BAL,2,all_offense,fringe,all_offense | fringe,282,strong,82.617188,79.687500,97.656250,90.625000,37.50000,-0.0866,0.0866,-0.0708,-0.1020,0.2778,0.0703,0.0510,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
8,BAL,3,early_down,fringe,early_down | fringe,219,strong,79.570312,74.218750,95.312500,89.583333,39.06250,-0.0855,0.0855,-0.0657,-0.1141,0.2162,0.0582,0.0491,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline
9,BUF,1,leading,own_territory,leading | own_territory,154,strong,87.890625,82.812500,96.875000,90.625000,78.12500,-0.0785,0.0785,-0.2334,-0.0476,0.2520,0.0938,0.0706,more_run_heavy_than_baseline,less_shotgun_than_baseline,slower_than_baseline,more_efficient_than_baseline


### Draft findings
- Adding field position sharpened the model. Some of the clearest average separation across teams shows up in contexts like `all_offense | own_territory`, `all_offense | fringe`, and `early_down | own_territory`.
- The field-zone rebuild changed the read. Teams that look similar in a broad situation can separate pretty quickly once ball location is added, especially in own territory, fringe space, and compressed scoring areas.
- Buffalo sits near the top of the model and shows one of its clearest signals in `leading_early_down | own_territory`, which gives me a concrete example of where a team should probably feel different from a default CPU profile.
- Splitting scoring space into `red_zone`, `goal_to_go`, and `goal_line` cleaned the model up. Those environments were being blended together before, and the outputs are stronger now that they’re separated.
- The best tuning candidates are the contexts where strong separation and strong sample overlap. That’s where I’d start if I were using this as a gameplay benchmark.